# Сквозной прогон Cinema DWH

В ноутбуке показана обработка данных от публикации в Kafka до оперативных и пакетных витрин ClickHouse. Сценарий включает полное архивирование в S3, обработку на Spark, исторический JOIN с SCD2 профилей, поздние сообщения, повторный расчёт, сверку RDD/DataFrame и общий отчёт с границей переключения T.

Для прогона использован набор из generate_data.py: режим demo, seed 42, 100 000 уникальных событий, 1 000 пользователей и три модельных дня с 1 по 4 января 2026 года (UTC). Генератор также создаёт повторные доставки и сообщения с поздним временем поступления. Проверки в ноутбуке сопоставляют результаты витрин с исходными данными.


## Подготовка инфраструктуры

Перед чистым прогоном сервисы запускаются из корня проекта. Команда docker compose down -v удаляет постоянные тома Docker с данными Kafka, S3, ClickHouse и PostgreSQL (включая метаданные Airflow); она используется только для подготовки нового прогона.

    docker compose down -v
    docker compose up -d
    docker compose exec airflow-scheduler python /opt/airflow/scripts/init_clickhouse.py
    docker compose --profile archive up -d archiver-events archiver-profiles

Архиваторы Kafka запущены до публикации тестовых сообщений. Ноутбук дополнительно проверяет, что в используемых таблицах и префиксах нет данных предыдущего прогона.


In [1]:
import json
import os
import re
import subprocess
import sys
import time
import uuid
from collections import Counter, defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import boto3

DATASET = "demo"
EVENTS = 100_000
USERS = 1_000
DAYS = 3
SEED = 42
START = datetime(2026, 1, 1, tzinfo=timezone.utc)
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
DATA_DIR = Path("/opt/airflow/notebooks") / f"generated_demo_{RUN_TAG}"
PERIOD_DATES = [(START + timedelta(days=i)).strftime("%Y-%m-%d") for i in range(DAYS)]
LOGICAL_DATES = [(START + timedelta(days=i + 1)).strftime("%Y-%m-%d") for i in range(DAYS)]
HORIZON_START = START.strftime("%Y-%m-%d %H:%M:%S")
HORIZON_END = (START + timedelta(days=DAYS)).strftime("%Y-%m-%d %H:%M:%S")

print(f"Dataset: {DATASET}; unique events requested: {EVENTS:,}; users: {USERS:,}; days: {DAYS}")
print(f"Model time range: [{HORIZON_START}, {HORIZON_END}) UTC")
print(f"Generated files will be saved under: {DATA_DIR}")

Dataset: demo; unique events requested: 100,000; users: 1,000; days: 3
Model time range: [2026-01-01 00:00:00, 2026-01-04 00:00:00) UTC
Generated files will be saved under: /opt/airflow/notebooks/generated_demo_20261125T163613Z_1d9712


## 1. Генерация набора данных

Скрипт generate_data.py запускается в режиме demo с фиксированными параметрами. Полученный манифест фиксирует параметры и контрольную сумму каталога; далее JSONL-файлы используются как независимый источник для сверки бизнес-показателей.


In [2]:
subprocess.run(
    [
        sys.executable,
        "/opt/airflow/scripts/generate_data.py",
        "--mode", "demo",
        "--events", str(EVENTS),
        "--users", str(USERS),
        "--days", str(DAYS),
        "--seed", str(SEED),
        "--start-time", "2026-01-01T00:00:00Z",
        "--output", str(DATA_DIR),
    ],
    check=True,
)

with (DATA_DIR / "manifest.json").open(encoding="utf-8") as source:
    manifest = json.load(source)

print(json.dumps(manifest, ensure_ascii=False, indent=2))

Создан /opt/airflow/notebooks/generated_demo_20261125T163613Z_1d9712: 101935 сообщений событий, 1992 сообщений профилей
{
  "schema_version": 1,
  "mode": "demo",
  "start_time": "2026-01-01T00:00:00Z",
  "seed": 42,
  "events": 100000,
  "users": 1000,
  "days": 3,
  "catalogue_sha256": "de951d1fc787b48960d97e73bc1be39d8357a8db7689692003da5e07a6d52762",
  "dataset_id": "281ecd0b7e78",
  "event_messages": 101935,
  "unique_events": 100000,
  "profile_messages": 1992,
  "unique_changes": 1991,
  "catalogue_rows": 200,
  "suggested_arrival_cutoff": "2026-01-04T00:00:00Z",
  "event_time_min": "2026-01-01T00:01:00Z",
  "event_time_max": "2026-01-03T23:59:54Z",
  "available_at_max": "2026-01-05T00:04:08Z"
}


Получаем, что создано 101 935 сообщений событий с 100 000 уникальными event_id, 1 992 сообщения профиля с 1 991 уникальным change_id и 200 строк каталога. Дополнительные сообщения отражают повторную доставку, но они не должны влиять на итоговые метрики.


In [3]:
def read_jsonl(path):
    with path.open(encoding="utf-8") as source:
        return [json.loads(line) for line in source if line.strip()]

events = read_jsonl(DATA_DIR / "events.jsonl")
profiles = read_jsonl(DATA_DIR / "profile_changes.jsonl")
catalogue = read_jsonl(DATA_DIR / "catalogue.jsonl")

events_by_id = {}
for row in events:
    events_by_id.setdefault(row["event_id"], row)

profiles_by_id = {}
for row in profiles:
    profiles_by_id.setdefault(row["change_id"], row)

assert len(events) == manifest["event_messages"]
assert len(events_by_id) == manifest["unique_events"] == EVENTS
assert len(profiles) == manifest["profile_messages"]
assert len(profiles_by_id) == manifest["unique_changes"]
assert len(events) > len(events_by_id)
assert len(profiles) > len(profiles_by_id)
assert len(catalogue) == manifest["catalogue_rows"]

print(f"Event messages: {len(events):,}; unique event_id: {len(events_by_id):,}")
print(f"Profile messages: {len(profiles):,}; unique change_id: {len(profiles_by_id):,}")
print(f"Catalogue rows: {len(catalogue):,}")

Event messages: 101,935; unique event_id: 100,000
Profile messages: 1,992; unique change_id: 1,991
Catalogue rows: 200


## 2. Проверка исходного состояния и подключений

Перед отправкой сообщений сначала проверим состояния ClickHouse и S3. На момент начала прогона в таблицах events_raw и profile_changes_raw было по 0 строк, архивные префиксы S3 отсутствовали, а пакетные витрины ещё не были созданы, что подтверждает изоляцию нового набора от предыдущих запусков.

In [4]:
S3_BUCKET = os.environ["S3_BUCKET"]
s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["S3_ENDPOINT"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=os.getenv("AWS_DEFAULT_REGION", "us-east-1"),
)

def ch_query(sql, timeout=60):
    params = urlencode({
        "user": os.environ["CLICKHOUSE_USER"],
        "password": os.environ["CLICKHOUSE_PASSWORD"],
    })
    request = Request(
        os.environ["CLICKHOUSE_URL"].rstrip("/") + "/?" + params,
        data=sql.encode("utf-8"),
        method="POST",
    )
    with urlopen(request, timeout=timeout) as response:
        return response.read().decode("utf-8").strip()

def ch_row(sql):
    result = ch_query(sql + " FORMAT TabSeparated")
    return result.splitlines()[0].split("\t") if result else []

def ch_int(sql):
    row = ch_row(sql)
    return int(row[0]) if row else 0

def list_s3_keys(prefix):
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=prefix):
        for item in page.get("Contents", []):
            yield item["Key"]

def archived_message_count(topic):
    prefix = f"raw/{topic}/"
    ranges = defaultdict(list)
    pattern = re.compile(r"partition=(\d+)/offset=(\d+)-(\d+)\.jsonl$")
    for key in list_s3_keys(prefix):
        match = pattern.search(key)
        if not match:
            continue
        partition, start_offset, end_offset = map(int, match.groups())
        ranges[partition].append((start_offset, end_offset))

    total = 0
    for partition, intervals in ranges.items():
        next_offset = 0
        for start_offset, end_offset in sorted(intervals):
            if start_offset != next_offset:
                raise RuntimeError(
                    f"S3 archive gap/overlap for {topic}, partition {partition}: "
                    f"expected offset {next_offset}, found {start_offset}"
                )
            if end_offset < start_offset:
                raise RuntimeError(f"Invalid offset range in {topic}: {start_offset}-{end_offset}")
            total += end_offset - start_offset + 1
            next_offset = end_offset + 1
    return total

raw_events = ch_int("SELECT count() FROM cinema.events_raw")
raw_profiles = ch_int("SELECT count() FROM cinema.profile_changes_raw")
old_event_archive_objects = any(list_s3_keys("raw/cinema.events/"))
old_profile_archive_objects = any(list_s3_keys("raw/cinema.profile_changes/"))
old_batch_tables = ch_int(
    "SELECT count() FROM system.tables "
    "WHERE database='cinema' AND name IN "
    "('batch_minute_metrics','batch_daily_profile_metrics','batch_daily_top5',"
    "'batch_search_word_frequencies','report_cutover')"
)

print({
    "events_raw": raw_events,
    "profile_changes_raw": raw_profiles,
    "existing_event_archive_objects": old_event_archive_objects,
    "existing_profile_archive_objects": old_profile_archive_objects,
    "existing_batch_tables": old_batch_tables,
})

if any((raw_events, raw_profiles, old_event_archive_objects,
        old_profile_archive_objects, old_batch_tables)):
    raise RuntimeError(
        "Обнаружены данные предыдущего прогона. "
        "Надо заново запустить проект с удалением всех предыдущих данных"
    )

{'events_raw': 0, 'profile_changes_raw': 0, 'existing_event_archive_objects': False, 'existing_profile_archive_objects': False, 'existing_batch_tables': 0}


## 3. Ранняя часть потока и её архивирование

Граница разделения сообщений взята из manifest.json: 2026-01-04T00:00:00Z. Сначала в Kafka отправляются сообщения с available_at раньше cutoff; сообщения на самой границе и после неё остаются для позднего этапа. Архиваторы S3 и потоковая загрузка ClickHouse работают одновременно.


In [5]:
def parse_utc(value):
    return datetime.fromisoformat(value.replace("Z", "+00:00")).astimezone(timezone.utc)

CUTOFF_TEXT = manifest["suggested_arrival_cutoff"]
CUTOFF = parse_utc(CUTOFF_TEXT)
early_event_messages = [row for row in events if parse_utc(row["available_at"]) < CUTOFF]
early_profile_messages = [row for row in profiles if parse_utc(row["available_at"]) < CUTOFF]
early_events_by_id = {}
for row in early_event_messages:
    early_events_by_id.setdefault(row["event_id"], row)

late_new_events_by_id = {}
for row in events:
    if parse_utc(row["available_at"]) >= CUTOFF and row["event_id"] not in early_events_by_id:
        late_new_events_by_id.setdefault(row["event_id"], row)

print(f"Arrival cutoff: {CUTOFF_TEXT}")
print(f"Early messages: events={len(early_event_messages):,}, profiles={len(early_profile_messages):,}")
print(f"New event IDs withheld until late stage: {len(late_new_events_by_id):,}")

subprocess.run(
    [
        sys.executable,
        "/opt/airflow/scripts/replay_kafka.py",
        "--dataset", str(DATA_DIR),
        "--bootstrap-servers", "kafka:29092",
        "--rate", "0",
        "--until-arrival", CUTOFF_TEXT,
    ],
    check=True,
)

Arrival cutoff: 2026-01-04T00:00:00Z
Early messages: events=96,689, profiles=1,905
New event IDs withheld until late stage: 5,138


%4|1795624646.507|GETPID|rdkafka#producer-1| [thrd:main]: Failed to acquire idempotence PID from broker kafka:29092/1: Broker: Coordinator load in progress: retrying


Поставлено в очередь: 98594; подтверждено: 98594; ошибки: 0; не завершено: 0


CompletedProcess(args=['/home/airflow/.local/bin/python', '/opt/airflow/scripts/replay_kafka.py', '--dataset', '/opt/airflow/notebooks/generated_demo_20261125T163613Z_1d9712', '--bootstrap-servers', 'kafka:29092', '--rate', '0', '--until-arrival', '2026-01-04T00:00:00Z'], returncode=0)

Получаем, что в ClickHouse загружено по 96 689 сообщений событий и 1 905 изменений профилей. Архивная проверка прошла по диапазонам Kafka offsets и не обнаружила пропусков или пересечений. При начальном подключении producer получил временный статус Coordinator load in progress и повторил запрос. 

В итоге подтверждена отправка всех 98 594 сообщений ранней части, ошибок и незавершённых отправок нет.

In [6]:
def wait_for_stage(expected_events, expected_profiles, timeout=900):
    deadline = time.monotonic() + timeout
    last = None
    while time.monotonic() < deadline:
        archived_events = archived_message_count("cinema.events")
        archived_profiles = archived_message_count("cinema.profile_changes")
        clickhouse_events = ch_int("SELECT count() FROM cinema.events_raw")
        clickhouse_profiles = ch_int("SELECT count() FROM cinema.profile_changes_raw")
        last = (archived_events, archived_profiles, clickhouse_events, clickhouse_profiles)

        if archived_events > expected_events or archived_profiles > expected_profiles:
            raise RuntimeError(f"В S3 больше сообщений, чем ожидалось: {last}")
        if clickhouse_events > expected_events or clickhouse_profiles > expected_profiles:
            raise RuntimeError(f"В ClickHouse больше сообщений, чем ожидалось: {last}")

        if (archived_events == expected_events and archived_profiles == expected_profiles
                and clickhouse_events == expected_events and clickhouse_profiles == expected_profiles):
            print({
                "archived_events": archived_events,
                "archived_profiles": archived_profiles,
                "clickhouse_event_messages": clickhouse_events,
                "clickhouse_profile_messages": clickhouse_profiles,
            })
            return

        print(f"Ожидание архиваторов/ClickHouse; текущие счётчики: {last}")
        time.sleep(5)

    raise TimeoutError(f"Сервисы не достигли ожидаемых счётчиков за {timeout}s; последнее: {last}")

wait_for_stage(len(early_event_messages), len(early_profile_messages))

{'archived_events': 96689, 'archived_profiles': 1905, 'clickhouse_event_messages': 96689, 'clickhouse_profile_messages': 1905}


## 4. Первая пакетная публикация и граница T

DAG обработал за промежуток времени [2026-01-01, 2026-01-02). После публикации граница переключения установлена в T = 2026-01-02 00:00:00 UTC: данные до T обслуживаются пакетной витриной, а данные от T - оперативной.

Для ранней части объединённый отчёт дал 71 269 просмотров и 180 495 012 секунд просмотра - ровно столько же, сколько независимый расчёт по уникальным входным событиям. До T опубликовано 16 265 строк поминутной витрины; от T в оперативном слое доступно 32 853 строки.

In [7]:
def run_dag(logical_date, label):
    command = [
        "airflow", "dags", "test", "cinema_batch_pipeline", logical_date,
        "--conf", json.dumps({"dataset": DATASET}),
    ]
    env = os.environ.copy()
    env["CINEMA_DAG_RETRIES"] = "0"
    print(f"Запуск {label}:", " ".join(command))
    try:
        result = subprocess.run(
            command, text=True, capture_output=True, timeout=1800, env=env
        )
    except subprocess.TimeoutExpired as error:
        stdout = error.stdout or ""
        stderr = error.stderr or ""
        if isinstance(stdout, bytes):
            stdout = stdout.decode("utf-8", errors="replace")
        if isinstance(stderr, bytes):
            stderr = stderr.decode("utf-8", errors="replace")
        print("\n".join((stdout + "\n" + stderr).splitlines()[-100:]))
        raise TimeoutError(f"Airflow DAG {label} не завершился за 1800 секунд") from error
    combined = (result.stdout or "") + "\n" + (result.stderr or "")
    if result.returncode:
        print("\n".join(combined.splitlines()[-100:]))
        raise RuntimeError(f"Airflow DAG завершился с кодом {result.returncode}")
    print("\n".join(combined.splitlines()[-45:]))

def report_cutover():
    return ch_query(
        "SELECT toString(argMax(cutover_time, updated_at)) "
        "FROM cinema.report_cutover WHERE report_name='views_minute'"
    )

def date_range_filter(column, start_text, end_text):
    return (
        f"{column} >= toDateTime('{start_text}', 'UTC') "
        f"AND {column} < toDateTime('{end_text}', 'UTC')"
    )

def batch_view_totals(start_text, end_text):
    condition = date_range_filter("minute", start_text, end_text)
    row = ch_row(
        "SELECT coalesce(sum(view_count),0), coalesce(sum(watch_time_seconds),0) "
        f"FROM cinema.batch_minute_metrics FINAL WHERE dataset='{DATASET}' AND {condition}"
    )
    return tuple(map(int, row))

def operational_view_totals(start_text, end_text):
    condition = date_range_filter("minute", start_text, end_text)
    row = ch_row(
        "SELECT coalesce(sum(view_count),0), coalesce(sum(watch_time_seconds),0) "
        f"FROM cinema.operational_minute_metrics WHERE {condition}"
    )
    return tuple(map(int, row))

def unified_view_totals(start_text, end_text):
    condition = date_range_filter("minute", start_text, end_text)
    row = ch_row(
        "SELECT coalesce(sum(view_count),0), coalesce(sum(watch_time_seconds),0) "
        f"FROM cinema.unified_minute_metrics WHERE {condition}"
    )
    return tuple(map(int, row))

def event_view_totals(rows, start_text, end_text):
    start_dt = datetime.strptime(start_text, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)
    end_dt = datetime.strptime(end_text, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)
    selected = [
        row for row in rows
        if row["event_type"] == "view"
        and start_dt <= parse_utc(row["event_time"]) < end_dt
    ]
    return len(selected), sum(int(row["watch_time_seconds"]) for row in selected)

run_dag(LOGICAL_DATES[0], "initial_day_1")

Запуск initial_day_1: airflow dags test cinema_batch_pipeline 2026-01-02 --conf {"dataset": "demo"}
	0 artifacts copied, 3 already retrieved (0kB/4ms)
26/11/25 16:39:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 16:39:19 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties

[Stage 0:>                                                         (0 + 4) / 33]

[Stage 0:=======>                                                  (4 + 4) / 33]

[Stage 0:==============>                                           (8 + 4) / 33]

[Stage 0:======================>                                  (13 + 4) / 33]

[Stage 0:==================================>                      (20 + 4) / 33]

[Stage 0:=============

In [8]:
first_day_end = (START + timedelta(days=1)).strftime("%Y-%m-%d %H:%M:%S")
expected_early_first_day = event_view_totals(
    list(early_events_by_id.values()),
    HORIZON_START,
    first_day_end,
)
actual_first_day_batch = batch_view_totals(HORIZON_START, first_day_end)

expected_cutover = first_day_end
actual_cutover = report_cutover()
assert actual_cutover == expected_cutover, (actual_cutover, expected_cutover)
assert actual_first_day_batch == expected_early_first_day, (
    actual_first_day_batch, expected_early_first_day
)

batch_before_t = ch_int(
    "SELECT count() FROM cinema.batch_minute_metrics FINAL "
    f"WHERE dataset='{DATASET}' AND minute < toDateTime('{actual_cutover}', 'UTC')"
)
operational_from_t = ch_int(
    "SELECT count() FROM cinema.operational_minute_metrics "
    f"WHERE minute >= toDateTime('{actual_cutover}', 'UTC') "
    f"AND minute < toDateTime('{HORIZON_END}', 'UTC')"
)

early_expected_all = event_view_totals(
    list(early_events_by_id.values()), HORIZON_START, HORIZON_END
)
early_unified = unified_view_totals(HORIZON_START, HORIZON_END)
assert early_unified == early_expected_all, (early_unified, early_expected_all)
assert batch_before_t > 0 and operational_from_t > 0

print({
    "T": actual_cutover,
    "batch_rows_before_T": batch_before_t,
    "operational_rows_from_T": operational_from_t,
    "unified_view_count_and_watch_seconds": early_unified,
    "expected_early_view_count_and_watch_seconds": early_expected_all,
})

{'T': '2026-01-02 00:00:00', 'batch_rows_before_T': 16265, 'operational_rows_from_T': 32853, 'unified_view_count_and_watch_seconds': (71269, 180495012), 'expected_early_view_count_and_watch_seconds': (71269, 180495012)}


## 5. Поздние сообщения и оперативные показатели

На втором этапе отправляются сообщения с available_at не раньше cutoff. В раннюю часть они не попадали, поэтому этот шаг проверяет приём опоздавших событий и изменений профиля. После отправки сверяются полные количества сообщений в Kafka-архиве S3 и сырых таблицах ClickHouse, а оперативные показатели сравниваются с исходным набором по уникальным событиям.


In [9]:
late_event_messages = [row for row in events if parse_utc(row["available_at"]) >= CUTOFF]
late_profile_messages = [row for row in profiles if parse_utc(row["available_at"]) >= CUTOFF]

subprocess.run(
    [
        sys.executable,
        "/opt/airflow/scripts/replay_kafka.py",
        "--dataset", str(DATA_DIR),
        "--bootstrap-servers", "kafka:29092",
        "--rate", "0",
        "--from-arrival", CUTOFF_TEXT,
    ],
    check=True,
)

wait_for_stage(len(events), len(profiles))

expected_full_views = event_view_totals(
    list(events_by_id.values()), HORIZON_START, HORIZON_END
)
actual_full_operational = operational_view_totals(HORIZON_START, HORIZON_END)
assert actual_full_operational == expected_full_views, (
    actual_full_operational, expected_full_views
)

print({
    "late_event_messages": len(late_event_messages),
    "late_profile_messages": len(late_profile_messages),
    "full_operational_view_count_and_watch_seconds": actual_full_operational,
    "expected_unique_view_count_and_watch_seconds": expected_full_views,
})

Поставлено в очередь: 5333; подтверждено: 5333; ошибки: 0; не завершено: 0
Ожидание архиваторов/ClickHouse; текущие счётчики: (99195, 1992, 96689, 1905)
{'archived_events': 101935, 'archived_profiles': 1992, 'clickhouse_event_messages': 101935, 'clickhouse_profile_messages': 1992}
{'late_event_messages': 5246, 'late_profile_messages': 87, 'full_operational_view_count_and_watch_seconds': (75130, 190463666), 'expected_unique_view_count_and_watch_seconds': (75130, 190463666)}


## 6. Исторический пересчёт и SCD2

В поздней части есть новые события за уже закрытый день и изменение профиля, поступившее после cutoff, но имеющее более ранний effective_at. DAG повторно обрабатывает 1 января. Проверяется, что новые уникальные просмотры вошли в пакетную витрину, дубликаты не изменили бизнес-метрики, а граница T не сдвинулась. Корректность временных интервалов SCD2 и исторического JOIN проверяется ниже независимыми сверками.


In [10]:
before_late_reprocess = batch_view_totals(HORIZON_START, first_day_end)
expected_late_first_day_delta = event_view_totals(
    [
        row for row in late_new_events_by_id.values()
        if START <= parse_utc(row["event_time"]) < START + timedelta(days=1)
    ],
    HORIZON_START,
    first_day_end,
)
assert expected_late_first_day_delta[0] > 0
assert late_profile_messages
assert any(parse_utc(row["effective_at"]) < CUTOFF for row in late_profile_messages)


t_before_late_reprocess = report_cutover()
run_dag(LOGICAL_DATES[0], "late_reprocess_day_1")
t_after_late_reprocess = report_cutover()
after_late_reprocess = batch_view_totals(HORIZON_START, first_day_end)

observed_delta = (
    after_late_reprocess[0] - before_late_reprocess[0],
    after_late_reprocess[1] - before_late_reprocess[1],
)
assert observed_delta == expected_late_first_day_delta, (
    observed_delta, expected_late_first_day_delta
)
assert t_after_late_reprocess == t_before_late_reprocess == first_day_end

print({
    "T_before_and_after_historical_reprocess": t_after_late_reprocess,
    "day_1_batch_before": before_late_reprocess,
    "day_1_batch_after": after_late_reprocess,
    "late_unique_day_1_view_delta": observed_delta,
    "expected_late_unique_day_1_view_delta": expected_late_first_day_delta,
})

Запуск late_reprocess_day_1: airflow dags test cinema_batch_pipeline 2026-01-02 --conf {"dataset": "demo"}
	0 artifacts copied, 3 already retrieved (0kB/4ms)
26/11/25 16:44:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 16:44:07 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties

[Stage 0:>                                                         (0 + 4) / 35]

[Stage 0:======>                                                   (4 + 4) / 35]

[Stage 0:===========>                                              (7 + 5) / 35]

[Stage 0:=============>                                            (8 + 4) / 35]

[Stage 0:===================>                                     (12 + 4) / 35]

[Stage 0:======

Поздний пересчёт увеличил показатель за 1 января с 23 681 до 24 958 просмотров, а суммарное время просмотра - с 59 750 566 до 62 917 666 секунд. Приращение составило 1 277 просмотров и 3 167 100 секунд, что совпало с независимым расчётом по новым уникальным событиям. Граница T осталась 2026-01-02 00:00:00 UTC.

Затем DAG опубликовал следующие закрытые сутки. После обработки всего трёхдневного диапазона T достигла 2026-01-04 00:00:00 UTC, а объединённый отчёт совпал с полным расчётом по источнику: 75 130 просмотров и 190 463 666 секунд.

In [11]:
for logical_date in LOGICAL_DATES[1:]:
    run_dag(logical_date, f"publish_{logical_date}")

final_expected_cutover = HORIZON_END
assert report_cutover() == final_expected_cutover, (
    report_cutover(), final_expected_cutover
)

final_unified = unified_view_totals(HORIZON_START, HORIZON_END)
assert final_unified == expected_full_views, (final_unified, expected_full_views)

print({
    "final_T": report_cutover(),
    "unified_view_count_and_watch_seconds": final_unified,
    "expected_all_unique_view_count_and_watch_seconds": expected_full_views,
})

Запуск publish_2026-01-03: airflow dags test cinema_batch_pipeline 2026-01-03 --conf {"dataset": "demo"}
[Stage 0:>                                                         (0 + 4) / 35]

[Stage 0:======>                                                   (4 + 4) / 35]

[Stage 0:=============>                                            (8 + 4) / 35]

[Stage 0:===================>                                     (12 + 4) / 35]

[Stage 0:==========================>                              (16 + 4) / 35]

[Stage 0:=======================================>                 (24 + 4) / 35]

[Stage 0:====================================================>    (32 + 3) / 35]

                                                                                

[Stage 6:================================>                        (20 + 4) / 35]

[Stage 6:====================================================>    (32 + 3) / 35]

[Stage 8:>                                                          (0 + 2)

## 7. Независимая проверка обработки и витрин

Пакетные результаты сверяются с исходными JSONL и с независимо построенной историей профилей. Проверки охватывают дедупликацию, соответствие каждого просмотра одному профилю и объекту каталога, непрерывность SCD2-интервалов, а также поминутную, дневную, топ-5 и поисковую витрины. Сравнения выполняются по ключам и значениям, а не только по общим суммам.


In [12]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("final-demo-requirements-check")
    .master("local[2]")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.hadoop.fs.s3a.endpoint", os.environ["S3_ENDPOINT"])
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

def parquet_period_paths(name):
    return [
        f"s3a://{S3_BUCKET}/processed/{DATASET}/period={period}/{name}"
        for period in PERIOD_DATES
    ]

processed_events = spark.read.parquet(*parquet_period_paths("events_deduplicated"))
enriched_views = spark.read.parquet(*parquet_period_paths("views_enriched"))
latest_scd2 = spark.read.parquet(
    f"s3a://{S3_BUCKET}/processed/{DATASET}/period={PERIOD_DATES[-1]}/profile_scd2"
)

processed_event_count = processed_events.count()
processed_event_duplicates = (
    processed_events.groupBy("event_id").count().filter(F.col("count") > 1).count()
)
enriched_view_count = enriched_views.count()
enriched_duplicates = (
    enriched_views.groupBy("event_id").count().filter(F.col("count") != 1).count()
)
missing_catalogue_matches = enriched_views.filter(F.col("category").isNull()).count()
missing_profile_matches = enriched_views.filter(F.col("country").isNull()).count()

scd_window = Window.partitionBy("user_id").orderBy("valid_from")
scd_checked = latest_scd2.withColumn("_next_from", F.lead("valid_from").over(scd_window))
scd_interval_errors = scd_checked.filter(
    ~F.col("valid_to").eqNullSafe(F.col("_next_from"))
).count()
scd_current_errors = (
    latest_scd2.groupBy("user_id")
    .agg(F.sum(F.col("is_current").cast("int")).alias("current_versions"))
    .filter(F.col("current_versions") != 1)
    .count()
)

assert processed_event_count == manifest["unique_events"]
assert processed_event_duplicates == 0
assert enriched_view_count == expected_full_views[0]
assert enriched_duplicates == 0
assert missing_catalogue_matches == 0
assert missing_profile_matches == 0
assert scd_interval_errors == 0
assert scd_current_errors == 0

print({
    "deduplicated_events": processed_event_count,
    "unique_view_rows": enriched_view_count,
    "duplicate_event_ids_in_processed_output": processed_event_duplicates,
    "duplicate_or_missing_profile_matches": enriched_duplicates,
    "views_without_catalogue_match": missing_catalogue_matches,
    "views_without_profile_match": missing_profile_matches,
    "SCD2_interval_errors": scd_interval_errors,
    "users_without_exactly_one_current_SCD2_version": scd_current_errors,
    "SCD2_users": latest_scd2.select("user_id").distinct().count(),
    "SCD2_versions": latest_scd2.count(),
})

:: loading settings :: url = jar:file:/home/airflow/.local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/airflow/.ivy2/cache
The jars for the packages stored in: /home/airflow/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7c6d5bbe-af39-47cc-a508-498ffd058402;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 149ms :: artifacts dl 8ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-------------------------------

{'deduplicated_events': 100000, 'unique_view_rows': 75130, 'duplicate_event_ids_in_processed_output': 0, 'duplicate_or_missing_profile_matches': 0, 'views_without_catalogue_match': 0, 'views_without_profile_match': 0, 'SCD2_interval_errors': 0, 'users_without_exactly_one_current_SCD2_version': 0, 'SCD2_users': 1000, 'SCD2_versions': 1902}


In [13]:
# Compare ClickHouse marts with metrics reconstructed from the generator output.
expected_minute = defaultdict(lambda: [0, 0])
for row in events_by_id.values():
    if row["event_type"] != "view":
        continue
    event_time = parse_utc(row["event_time"])
    minute = event_time.replace(second=0, microsecond=0).strftime("%Y-%m-%d %H:%M:%S")
    key = (minute, int(row["content_id"]))
    expected_minute[key][0] += 1
    expected_minute[key][1] += int(row["watch_time_seconds"])

actual_minute_rows = ch_query(
    "SELECT toString(minute), content_id, view_count, watch_time_seconds "
    "FROM cinema.batch_minute_metrics FINAL "
    f"WHERE dataset='{DATASET}' ORDER BY minute, content_id FORMAT TabSeparated"
).splitlines()
actual_minute = [
    (row.split("\t")[0], int(row.split("\t")[1]),
     int(row.split("\t")[2]), int(row.split("\t")[3]))
    for row in actual_minute_rows if row
]
expected_minute_rows = sorted(
    (minute, content_id, values[0], values[1])
    for (minute, content_id), values in expected_minute.items()
)
assert actual_minute == expected_minute_rows

expected_words = Counter()
for row in events_by_id.values():
    if row["event_type"] != "search":
        continue
    day = parse_utc(row["event_time"]).date().isoformat()
    for word in (row.get("query") or "").lower().split():
        if len(word) > 4:
            expected_words[(day, word)] += 1

actual_word_rows = ch_query(
    "SELECT toString(event_date), word, search_count "
    "FROM cinema.batch_search_word_frequencies FINAL "
    f"WHERE dataset='{DATASET}' ORDER BY event_date, word FORMAT TabSeparated"
).splitlines()
actual_words = [
    (row.split("\t")[0], row.split("\t")[1], int(row.split("\t")[2]))
    for row in actual_word_rows if row
]
expected_word_rows = sorted(
    (day, word, count) for (day, word), count in expected_words.items()
)
assert actual_words == expected_word_rows

daily_content_counts = Counter()
for row in events_by_id.values():
    if row["event_type"] == "view":
        day = parse_utc(row["event_time"]).date().isoformat()
        daily_content_counts[(day, int(row["content_id"]))] += 1

expected_top5 = []
for day in sorted({key[0] for key in daily_content_counts}):
    ranked = sorted(
        ((content_id, count)
         for (row_day, content_id), count in daily_content_counts.items()
         if row_day == day),
        key=lambda item: (-item[1], item[0]),
    )[:5]
    expected_top5.extend(
        (day, content_id, count, rank)
        for rank, (content_id, count) in enumerate(ranked, start=1)
    )

actual_top5_rows = ch_query(
    "SELECT toString(event_date), content_id, view_count, rank "
    "FROM cinema.batch_daily_top5 FINAL "
    f"WHERE dataset='{DATASET}' ORDER BY event_date, rank FORMAT TabSeparated"
).splitlines()
actual_top5 = [
    (row.split("\t")[0], int(row.split("\t")[1]),
     int(row.split("\t")[2]), int(row.split("\t")[3]))
    for row in actual_top5_rows if row
]
assert actual_top5 == expected_top5

print({
    "minute_mart_rows_exactly_matching_source": len(actual_minute),
    "search_word_rows_exactly_matching_source": len(actual_words),
    "daily_top5_rows_exactly_matching_source": len(actual_top5),
})

{'minute_mart_rows_exactly_matching_source': 51485, 'search_word_rows_exactly_matching_source': 39, 'daily_top5_rows_exactly_matching_source': 15}


In [14]:
import bisect

profile_changes_by_user = defaultdict(list)
for change in profiles_by_id.values():
    profile_changes_by_user[int(change["user_id"])].append(change)

profile_versions_by_user = {}
for user_id, changes in profile_changes_by_user.items():
    changes.sort(key=lambda row: (parse_utc(row["effective_at"]), row["change_id"]))
    versions = []
    previous_state = None
    for change in changes:
        state = (change["country"], change["subscription_tier"])
        if state != previous_state:
            versions.append((parse_utc(change["effective_at"]), *state))
            previous_state = state
    profile_versions_by_user[user_id] = versions

source_profile_metrics = defaultdict(lambda: [0, 0])
for event in events_by_id.values():
    if event["event_type"] != "view":
        continue
    event_time = parse_utc(event["event_time"])
    versions = profile_versions_by_user[int(event["user_id"])]
    starts = [version[0] for version in versions]
    version_index = bisect.bisect_right(starts, event_time) - 1
    assert version_index >= 0, (event["event_id"], event["user_id"], event["event_time"])
    _, country, tier = versions[version_index]
    key = (event_time.date().isoformat(), country, tier)
    source_profile_metrics[key][0] += 1
    source_profile_metrics[key][1] += int(event["watch_time_seconds"])

source_profile_rows = sorted(
    (day, country, tier, values[0], values[1])
    for (day, country, tier), values in source_profile_metrics.items()
)

profile_expected_rows = (
    enriched_views
    .withColumn("event_date", F.to_date("event_time"))
    .groupBy("event_date", "country", "subscription_tier")
    .agg(
        F.countDistinct("event_id").alias("view_count"),
        F.sum("watch_time_seconds").alias("watch_time_seconds"),
    )
    .orderBy("event_date", "country", "subscription_tier")
    .collect()
)
profile_expected = [
    (row["event_date"].isoformat(), row["country"], row["subscription_tier"],
     int(row["view_count"]), int(row["watch_time_seconds"]))
    for row in profile_expected_rows
]

profile_actual_rows = ch_query(
    "SELECT toString(event_date), country, subscription_tier, view_count, watch_time_seconds "
    "FROM cinema.batch_daily_profile_metrics FINAL "
    f"WHERE dataset='{DATASET}' "
    "ORDER BY event_date, country, subscription_tier FORMAT TabSeparated"
).splitlines()
profile_actual = [
    (row.split("\t")[0], row.split("\t")[1], row.split("\t")[2],
     int(row.split("\t")[3]), int(row.split("\t")[4]))
    for row in profile_actual_rows if row
]
assert profile_actual == profile_expected
assert profile_actual == source_profile_rows

minute_totals = (
    sum(row[2] for row in actual_minute),
    sum(row[3] for row in actual_minute),
)
profile_totals = (
    sum(row[3] for row in profile_actual),
    sum(row[4] for row in profile_actual),
)
assert minute_totals == profile_totals == expected_full_views

print({
    "daily_country_tier_rows_matching_historical_join": len(profile_actual),
    "daily_country_tier_rows_matching_source_profile_history": len(source_profile_rows),
    "minute_and_country_tier_totals": minute_totals,
    "expected_unique_view_count_and_watch_seconds": expected_full_views,
})

{'daily_country_tier_rows_matching_historical_join': 36, 'daily_country_tier_rows_matching_source_profile_history': 36, 'minute_and_country_tier_totals': (75130, 190463666), 'expected_unique_view_count_and_watch_seconds': (75130, 190463666)}


Итог по нескольким последним ячейкам: обработано ровно 100 000 уникальных событий, получено 75 130 строк просмотров без дубликатов. Для всех просмотров найдены соответствия в каталоге и в истории профилей. Для 1 000 пользователей построено 1 902 версии SCD2; нарушений границ интервалов и числа текущих версий не обнаружено.

С пакетными витринами совпали независимые расчёты по источнику: 51 485 строк поминутной витрины, 39 строк частот поисковых слов и 15 строк дневного топ-5. Дневная витрина по стране и тарифу содержит 36 групп и совпала как с историческим JOIN Spark, так и с расчётом по исходной истории профилей. Общие суммы обеих детализаций равны 75 130 просмотрам и 190 463 666 секундам.

## 8. Ключи, метаданные и сверка RDD/DataFrame

Дополнительно проверяются уникальность бизнес-ключей пакетных таблиц, метаданные запусков Airflow, наличие каталога в S3 и совпадение результатов двух реализаций Spark.


In [15]:
mart_keys = {
    "batch_minute_metrics": "minute, content_id, dataset",
    "batch_daily_profile_metrics": "event_date, country, subscription_tier, dataset",
    "batch_daily_top5": "event_date, content_id, dataset",
    "batch_search_word_frequencies": "event_date, word, dataset",
}
key_check_results = {}
for table, keys in mart_keys.items():
    key_check_results[table] = ch_int(
        "SELECT count() FROM ("
        f"SELECT {keys} FROM cinema.{table} FINAL "
        f"WHERE dataset='{DATASET}' GROUP BY {keys} HAVING count() > 1)"
    )
assert all(value == 0 for value in key_check_results.values())

metadata = ch_row(
    "SELECT count(), uniqExact(_dag_id), uniqExact(_run_id), "
    "uniqExact(_data_interval_start), uniqExact(_data_interval_end), "
    "countIf(_dag_id != 'cinema_batch_pipeline' OR _run_id = '' OR _task_id = '') "
    "FROM cinema.batch_minute_metrics FINAL "
    f"WHERE dataset='{DATASET}'"
)
metadata_summary = tuple(map(int, metadata))
assert metadata_summary[0] == len(actual_minute)
assert metadata_summary[1] == 1
assert metadata_summary[2] >= DAYS
assert metadata_summary[3] == DAYS
assert metadata_summary[4] == DAYS
assert metadata_summary[5] == 0

rdd_word_rows = spark.read.parquet(
    f"s3a://{S3_BUCKET}/checks/{DATASET}/rdd_dataframe/search_words"
).count()
rdd_top5_rows = spark.read.parquet(
    f"s3a://{S3_BUCKET}/checks/{DATASET}/rdd_dataframe/daily_top5"
).count()
assert rdd_word_rows == len(expected_words)
assert rdd_top5_rows == len(expected_top5)

catalogue_body = s3.get_object(
    Bucket=S3_BUCKET, Key=f"source/{DATASET}/catalogue.jsonl"
)["Body"]
archived_catalogue = [
    json.loads(line) for line in catalogue_body.iter_lines() if line
]
assert archived_catalogue == catalogue
catalogue_body.close()

print({
    "duplicate_business_keys": key_check_results,
    "minute_mart_metadata": {
        "rows": metadata_summary[0],
        "dag_ids": metadata_summary[1],
        "run_ids": metadata_summary[2],
        "interval_starts": metadata_summary[3],
        "interval_ends": metadata_summary[4],
        "invalid_or_missing_metadata_rows": metadata_summary[5],
    },
    "catalogue_rows_archived_in_s3": len(archived_catalogue),
    "RDD_DataFrame_compared_search_rows": rdd_word_rows,
    "RDD_DataFrame_compared_top5_rows": rdd_top5_rows,
})

{'duplicate_business_keys': {'batch_minute_metrics': 0, 'batch_daily_profile_metrics': 0, 'batch_daily_top5': 0, 'batch_search_word_frequencies': 0}, 'minute_mart_metadata': {'rows': 51485, 'dag_ids': 1, 'run_ids': 3, 'interval_starts': 3, 'interval_ends': 3, 'invalid_or_missing_metadata_rows': 0}, 'catalogue_rows_archived_in_s3': 200, 'RDD_DataFrame_compared_search_rows': 39, 'RDD_DataFrame_compared_top5_rows': 15}


По четырём пакетным витринам не найдено повторяющихся бизнес-ключей. Все 51 485 строк поминутной витрины содержат корректные метаданные: один DAG, три запуска и три закрытых интервала; некорректных или пустых значений нет. В S3 сохранены все 200 строк каталога.

Задача сравнения RDD и DataFrame завершилась успешно: реализации дали одинаковые результаты для частот слов (39 строк) и дневного топ-5 (15 строк).

## 9. Идемпотентность повторного запуска

Последний уже опубликованный интервал запускается повторно на тех же входных данных. Сравниваются число строк и бизнес-суммы четырёх пакетных витрин, а также значение T.

In [16]:
def batch_snapshot():
    queries = {
        "minute": (
            "SELECT count(), sum(view_count), sum(watch_time_seconds) "
            "FROM cinema.batch_minute_metrics FINAL"
        ),
        "daily_profile": (
            "SELECT count(), sum(view_count), sum(watch_time_seconds) "
            "FROM cinema.batch_daily_profile_metrics FINAL"
        ),
        "daily_top5": (
            "SELECT count(), sum(view_count) FROM cinema.batch_daily_top5 FINAL"
        ),
        "search_words": (
            "SELECT count(), sum(search_count) FROM cinema.batch_search_word_frequencies FINAL"
        ),
    }
    result = {}
    for name, sql in queries.items():
        row = ch_row(sql + f" WHERE dataset='{DATASET}'")
        result[name] = tuple(map(int, row))
    return result

snapshot_before_rerun = batch_snapshot()
t_before_rerun = report_cutover()
run_dag(LOGICAL_DATES[-1], "idempotent_rerun_last_day")
snapshot_after_rerun = batch_snapshot()
t_after_rerun = report_cutover()

assert snapshot_after_rerun == snapshot_before_rerun
assert t_after_rerun == t_before_rerun == HORIZON_END

print({
    "batch_metrics_unchanged": snapshot_after_rerun == snapshot_before_rerun,
    "T_before_and_after": (t_before_rerun, t_after_rerun),
    "batch_snapshot": snapshot_after_rerun,
})

Запуск idempotent_rerun_last_day: airflow dags test cinema_batch_pipeline 2026-01-04 --conf {"dataset": "demo"}
[Stage 0:===================>                                     (12 + 4) / 35]

[Stage 0:========================>                                (15 + 4) / 35]

[Stage 0:================================>                        (20 + 4) / 35]

[Stage 0:=============================================>           (28 + 4) / 35]

[Stage 0:====================================================>    (32 + 3) / 35]

                                                                                

[Stage 6:================================>                        (20 + 4) / 35]

[Stage 6:=============================================>           (28 + 4) / 35]

[Stage 8:>                                                          (0 + 2) / 2]

                                                                                

[Stage 10:=========================>                              (1

## Итоги прогона

Сквозной прогон прошёл от Kafka до ClickHouse. Архив и сырые таблицы содержат все 101 935 сообщений событий и 1 992 сообщения профиля; каталог из 200 строк сохранён в S3. После дедупликации учтены 100 000 событий, включая 75 130 просмотров и 190 463 666 секунд просмотра.

| Проверка | Результат |
| --- | --- |
| Ранний пакетный и оперативный слои | Отчёт совпал с источником до и после поздней загрузки; T достигла 2026-01-04 00:00:00 UTC |
| Поздний пересчёт 1 января | Приращение 1 277 просмотров и 3 167 100 секунд совпало с независимым расчётом; T не изменилась |
| SCD2 и исторические JOIN | 1 000 пользователей, 1 902 версии; пропусков соответствий и ошибок интервалов не обнаружено |
| Пакетные витрины | 51 485 поминутных строк, 36 дневных групп, 15 строк топ-5 и 39 строк поисковых частот; сверки с источником прошли |
| RDD и DataFrame | Частоты поисковых слов и дневной топ-5 совпали построчно |
| Повторная публикация | Бизнес-снимок витрин не изменился; T осталась 2026-01-04 00:00:00 UTC |